# InceptionResNetV2

##  Step 1: Import Necessary libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

## Step 2: CHANGE ONLY THIS SECTION

In [6]:
IMAGE_SIZE = (224, 224)          # <- Change input size
BATCH_SIZE = 32
EPOCHS = 20

TRAIN_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\train_data"
VAL_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\val_data"
TEST_DIR=r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data"
NUM_CLASSES = 5                  # <- Number of classes

BASE_MODEL_NAME ="InceptionResNetV2"    # <- Options: VGG16, ResNet50, MobileNetV2, etc

FREEZE_LAYERS = True             # <- Freeze base model
FINE_TUNE_AT = None              # <- Set layer index to unfreeze later

## Step 3: Data Preparation || Data Pipeline

In [4]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   horizontal_flip=True,
                                   zoom_range=0.2)

val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = train_datagen.flow_from_directory(TEST_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical',
                                               shuffle=False)
train_data = train_datagen.flow_from_directory(TRAIN_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical')

val_data = val_datagen.flow_from_directory(VAL_DIR,
                                           target_size=IMAGE_SIZE,
                                           batch_size=BATCH_SIZE,
                                           class_mode='categorical')

Found 15 images belonging to 5 classes.
Found 50 images belonging to 5 classes.
Found 9 images belonging to 5 classes.


## Step 4: Model Building: Load PreTrained Model

In [5]:
def get_base_model(name):
    if name == "VGG16":
        return tf.keras.applications.VGG16(weights='imagenet',
                                           include_top=False,
                                           input_shape=(*IMAGE_SIZE, 3))
    elif name == "ResNet50":
        return tf.keras.applications.ResNet50(weights='imagenet',
                                              include_top=False,
                                              input_shape=(*IMAGE_SIZE, 3))
    elif name == "MobileNetV2":
        return tf.keras.applications.MobileNetV2(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(*IMAGE_SIZE, 3))
    elif name == "InceptionResNetV2":
        return tf.keras.applications.InceptionResNetV2(weights='imagenet',
                                                       include_top=False,
                                                       input_shape=(*IMAGE_SIZE, 3))

base_model = get_base_model(BASE_MODEL_NAME)


219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 20s 0us/step


### ====================================================================

# The Most Important 2 Steps in Transfer Learning: 
## 1. FREEZE BASE MODEL

In [7]:
if FREEZE_LAYERS:
    for layer in base_model.layers:
        layer.trainable = False

## 2. ADD CUSTOM HEAD

In [8]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 111, 111, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 111, 111, 32)      │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 111, 111, 32)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 109, 109, 32)      │           9,216 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 109, 109, 32)      │              96 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 109, 109, 32)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 109, 109, 64)      │          18,432 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 109, 109, 64)      │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 109, 109, 64)      │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 54, 54, 64)        │               0 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 54, 54, 80)        │             240 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 54, 54, 80)        │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 52, 52, 192)       │         138,240 │ activation_3[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 54,731,493 (208.78 MB)

 Trainable params: 394,757 (1.51 MB)

 Non-trainable params: 54,336,736 (207.28 MB)

### ====================================================================

### Step 4: Model Building Continues..It's COMPILATION Time.

In [9]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 5: Model Training | Model Evaluation | Model Testing

In [10]:
history = model.fit(train_data,
                    validation_data=val_data,
                    epochs=EPOCHS)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 7s/step - accuracy: 0.2200 - loss: 2.2769 - val_accuracy: 0.2222 - val_loss: 2.0159
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - accuracy: 0.2400 - loss: 2.1296 - val_accuracy: 0.2222 - val_loss: 1.9068
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - accuracy: 0.2800 - loss: 2.0390 - val_accuracy: 0.2222 - val_loss: 1.8464
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - accuracy: 0.1200 - loss: 2.0052 - val_accuracy: 0.3333 - val_loss: 1.8157
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - accuracy: 0.2600 - loss: 1.8998 - val_accuracy: 0.3333 - val_loss: 1.7948
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.3800 - loss: 1.6081 - val_accuracy: 0.3333 - val_loss: 1.7822
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.4000 - loss: 1.6232 - val_accuracy: 0.3333 - val_loss: 1.7691
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step - accuracy: 0.3200 - loss: 1.5483 - val_accuracy: 0.3333 - val_loss: 1.7427
Epoch 9/20
2/2 

##  (OPTIONAL Step): FINE-TUNING with new weights(NOT SUGGESTED)

In [11]:
if FINE_TUNE_AT is not None:
    for layer in base_model.layers[FINE_TUNE_AT:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("Starting Fine-Tuning...")

    history_fine = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5
    )

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 111, 111, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 111, 111, 32)      │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 111, 111, 32)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 109, 109, 32)      │           9,216 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 109, 109, 32)      │              96 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 109, 109, 32)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 109, 109, 64)      │          18,432 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 109, 109, 64)      │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 109, 109, 64)      │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 54, 54, 64)        │               0 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 54, 54, 80)        │             240 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 54, 54, 80)        │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 52, 52, 192)       │         138,240 │ activation_3[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 55,521,009 (211.80 MB)

 Trainable params: 394,757 (1.51 MB)

 Non-trainable params: 54,336,736 (207.28 MB)

 Optimizer params: 789,516 (3.01 MB)

## Export the Intelligence File.

### Question: How to use this file for Prediction?

In [14]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\Vijay\images (4).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step
Predicted Person: DQ


In [13]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\DQ\images (12).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step
Predicted Person: DQ


# THE END!